# SILMA TTS v1 — دبلجة عربية واستنساخ صوت على Colab

هذا الدفتر يستخدم SILMA TTS v1، وهو نموذج عربي/إنجليزي خفيف مع استنساخ صوت صفري من مرجع قصير. نبدأ بفحص GPU، ثم نثبت الاعتمادات دون استبدال PyTorch الجاهز في Colab، ونولّد مقطعًا واحدًا فقط للتحقق قبل الدبلجة الكاملة.

استخدم فقط مراجع صوتية لديك إذن صريح لاستعمالها، وصرّح بأن الصوت مولّد بالذكاء الاصطناعي عند مشاركة الناتج.
    


In [ ]:
import os, re, shutil, subprocess, torch
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True,max_split_size_mb:128'
print('torch:', torch.__version__)
print('torch CUDA build:', torch.version.cuda)
print('cuda available:', torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError('PyTorch لا يرى CUDA. اختر GPU runtime في Colab ثم أعد تشغيل الخلية.')
gpu = subprocess.run(['nvidia-smi','--query-gpu=name,memory.total,memory.free','--format=csv,noheader'], capture_output=True, text=True, check=True)
print(gpu.stdout.strip())
print('GPU:', torch.cuda.get_device_name(0))
usage = shutil.disk_usage('/content')
print('disk free GiB:', round(usage.free/2**30, 1), '/', round(usage.total/2**30, 1))
    


In [ ]:
%cd /content
!apt-get update -qq && apt-get install -y -qq ffmpeg libsox-dev
!pip -q install --upgrade pip
# SILMA 1.0.5 requires NumPy <= 1.26.4; install only this compatible dependency.
!pip -q install --no-deps numpy==1.26.4
# Install SILMA without letting pip replace Colab's CUDA-enabled torch stack.
!pip -q install --no-deps silma-tts==1.0.5
!pip -q install cached_path click ema_pytorch hydra-core librosa matplotlib pydub safetensors soundfile tomli torchdiffeq tqdm transformers transformers_stream_generator unidecode vocos x_transformers nemo_text_processing==1.1.0 catt_tashkeel==1.0.2
!rm -rf /content/dub22
!git clone --depth 1 https://github.com/dhiyaddineb-hue/dub22.git /content/dub22
print('تم تثبيت SILMA وworkspace دون استبدال PyTorch.')
    


In [ ]:
import torch, subprocess, shutil
print('post-install torch:', torch.__version__)
print('post-install CUDA:', torch.version.cuda)
print('post-install cuda available:', torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError('فقدت CUDA بعد التثبيت؛ لن أحمّل النموذج.')
print(subprocess.run(['nvidia-smi','--query-gpu=name,memory.total,memory.free','--format=csv,noheader'], capture_output=True, text=True, check=True).stdout.strip())
usage = shutil.disk_usage('/content')
print('disk free GiB:', round(usage.free/2**30, 1))
from silma_tts.api import SilmaTTS
print('SILMA API import: OK')
    


In [ ]:
from pathlib import Path
from google.colab import files

input_video = Path('/content/dub22/assets/input/new_job/source.mp4')
use_upload = False
if use_upload:
    uploaded = files.upload()
    if len(uploaded) != 1:
        raise RuntimeError('ارفع فيديو واحدًا فقط.')
    name = next(iter(uploaded))
    input_video = Path('/content/dub22/assets/input/colab_job') / name
    input_video.parent.mkdir(parents=True, exist_ok=True)
    input_video.write_bytes(uploaded[name])
if not input_video.exists():
    raise FileNotFoundError(input_video)
print('الفيديو:', input_video)
    


## اختبار مقطع واحد

لا تشغّل الدبلجة الكاملة قبل الاستماع إلى هذا الملف. الإعداد الافتراضي يعطّل التطبيع التلقائي والتشكيل الآلي لتقليل الاعتمادات وزمن البدء؛ يمكن تفعيلهما لاحقًا إذا كانت النتيجة العربية تحتاج ذلك.
    


In [ ]:
import json, subprocess, sys
from pathlib import Path

manifest_path = Path('/content/dub22/manifests/new_job/dialogue_ar_fireredtts3.json')
manifest = json.loads(manifest_path.read_text(encoding='utf-8'))
smoke_manifest_path = Path('/content/dub22/manifests/new_job/dialogue_ar_silma_smoke.json')
smoke_manifest = dict(manifest)
smoke_manifest['voice_mode'] = 'silma_tts_v1_zero_shot_clone'
smoke_manifest['segments'] = manifest['segments'][:1]
smoke_manifest_path.write_text(json.dumps(smoke_manifest, ensure_ascii=False, indent=2), encoding='utf-8')
smoke_output = Path('/content/dub22/outputs/new_job/arabic_dub_silma_smoke_line_01.mp4')
command = [
    sys.executable, '/content/dub22/scripts/silma_dub.py',
    '--input', str(input_video),
    '--manifest', str(smoke_manifest_path),
    '--output', str(smoke_output),
    '--workdir', '/content/dub22/assets/silma/colab_smoke',
    '--duration', '59.4',
    '--limit-segments', '1',
    '--nfe-step', '16',
    '--cfg-strength', '2.0',
]
print('بدء اختبار SILMA للمقطع الأول فقط...')
result = subprocess.run(command, cwd='/content/dub22', text=True)
if result.returncode != 0 or not smoke_output.exists() or smoke_output.stat().st_size == 0:
    raise RuntimeError('فشل اختبار SILMA؛ لن أشغّل الدبلجة الكاملة.')
print('نجح الاختبار تقنيًا:', smoke_output)
print(subprocess.run(['ffprobe','-v','error','-show_entries','format=duration','-of','default=noprint_wrappers=1:nokey=1',str(smoke_output)], capture_output=True, text=True, check=True).stdout.strip(), 'seconds')
    


In [ ]:
from IPython.display import Video, display
from google.colab import files
smoke_output = Path('/content/dub22/outputs/new_job/arabic_dub_silma_smoke_line_01.mp4')
display(Video(str(smoke_output), embed=False))
# Download only after listening/approving the one-line sample.
# files.download(str(smoke_output))
    


## الدبلجة الكاملة

شغّل الخلية التالية فقط بعد التحقق من طبيعية الصوت وهوية المتحدث والمزامنة. إذا ظهرت أخطاء ذاكرة أو كانت الهوية الصوتية غير مقنعة، توقف عند العينة ولا تعتبر المسار ناجحًا.
    


In [ ]:
import subprocess, sys
from pathlib import Path
output_video = Path('/content/dub22/outputs/new_job/arabic_dub_silma.mp4')
command = [
    sys.executable, '/content/dub22/scripts/silma_dub.py',
    '--input', str(input_video),
    '--manifest', '/content/dub22/manifests/new_job/dialogue_ar_fireredtts3.json',
    '--output', str(output_video),
    '--workdir', '/content/dub22/assets/silma/colab_full',
    '--duration', '59.4',
    '--nfe-step', '16',
    '--cfg-strength', '2.0',
]
print('بدء الدبلجة الكاملة بـSILMA...')
result = subprocess.run(command, cwd='/content/dub22', text=True)
if result.returncode != 0 or not output_video.exists() or output_video.stat().st_size == 0:
    raise RuntimeError('فشلت الدبلجة الكاملة بـSILMA.')
print('تم الإنتاج:', output_video)
    


In [ ]:
from IPython.display import Video, display
from google.colab import files
output_video = Path('/content/dub22/outputs/new_job/arabic_dub_silma.mp4')
display(Video(str(output_video), embed=False))
files.download(str(output_video))
    
